# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
%load_ext dotenv
%dotenv ../05_src/.secrets

import sys
sys.path.append('../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Create the loader and load PDF

loader = PyPDFLoader("documents\managing_oneself.pdf")
docs = loader.load()

# Join all pages into one string 

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from utils.clients import get_client
import os
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

In [ ]:
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field


llm = init_chat_model(MODEL, 
                      model_provider="openai",
                       base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                       default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
                      )

# Setup class for structured output 

class DocAbstract(BaseModel): 
    author: str = Field (description = "The author of the article")
    title: str = Field (description = "The title of the article")
    relevance: str = Field (description = " A statement, no longer than a paragraph, that explains why is the article relevant for an AI professional in their professional development")
    summary: str = Field (description = "A concise and succinct summary, no longer than 1000 tokens")
    tone: str = Field (description = "The writing style used to produce the summary")
    input_tokens: int
    output_tokens: int

structured_llm = llm.with_structured_output(DocAbstract)

In [ ]:
# Define tone 

tone = "Journalistic"

# Instructions

instructions = f"""
    You are a writer at a popular news outlet. 
    Your job is to summarize documents using a given tone, for teleprompters so that news reporters can present them.
    Keep your work factual and objective, with not emotional jargon. 

    Tone: 
    {tone}  
"""

# Prompt

prompt = f"""
    Analyze the following document for the following:  
    - author, 
    - title, 
    - the tone used to produce the summary
    - a short paragraph about why the content of the document is relevant for an AI professional in their professional development, 
    - a concise and succinct summary of the document 

    Document: 
    {document_text}
"""

In [ ]:
from pprint import pprint

# Invoke the LLM to generate a structured summary of the document using the instructions and prompt

doc_abstract_output = structured_llm.invoke([
    ("developer", instructions),
    ("user", prompt)
])

pprint(doc_abstract_output.model_dump())

{'author': 'Peter F. Drucker',
 'input_tokens': 4383,
 'output_tokens': 924,
 'relevance': "Understanding one's strengths, weaknesses, and values is "
              'critical for AI professionals as they navigate a rapidly '
              'evolving knowledge economy. This article emphasizes '
              'self-management, which is essential for adapting to continuous '
              'changes in technology and workplace dynamics, ultimately '
              'enabling professionals to maximize their contributions to their '
              'organizations.',
 'summary': 'In "Managing Oneself", Peter Drucker asserts that in today\'s '
            'knowledge economy, individuals must take charge of their own '
            'careers, effectively acting as their own CEOs. To achieve '
            'personal and professional success, one should develop a deep '
            'understanding of their strengths, work styles, values, and the '
            'environments they thrive in. Drucker provides 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

In [ ]:
# Evaluation test case

test_case = LLMTestCase(input = document_text, actual_output = doc_abstract_output.summary)

# Evaluation 
# Summarization  

metric = SummarizationMetric(
    threshold=0.7,
    model=model,
    assessment_questions=[
        "Does the summary accurately capture the main purpose of the original document?", 
        #"Does the summary include the most important ideals and key points from the document?", 
        "Is the summary factually consistent with the original document without introducing new content?", 
        #"Is the summary concise while still providing enough detail to understand the document?", 
        "Does the summary explain Drucker's argument that individuals should actively manage their own careers?",
        "Does the summary mention the importance of understanding one's strengths, values, or work style?",
        "Would someone who has not read the article understand its key message after reading the summary?"
    ]
)

# Clarity

clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Is the summary written in clear and understandable language?",
        "Is the information presented in a logical and well-organized order?",
        "Are technical terms or jargon explained or used appropriately for the intended audience?",
        #"Does the summary avoid unnecessary repetition or overly complex sentences?",
        #"Can an average reader easily understand the main message after reading the summary?"
        "Does the summary make Drucker's recommendations easy for the reader to understand?",
        "Can an average reader easily identify the main takeaways from the summary?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

# Tonality 

tonality = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Does the summary maintain a professional news reporter writing style throughout?",
        "Is the language objective and free from personal opinions or bias?",
        "Does the summary avoid overly emotional or exaggerated language?",
        "Is the tone consistent from beginning to end?",
        "Is the writing appropriate for a general audience while remaining professional?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

# Safety 

safety = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Does the summary avoid generating false or misleading information that is not supported by the original document?",
        "Does the summary avoid offensive, hateful, or discriminatory language?",
        "Does the summary avoid revealing sensitive or private information that is not present in the original document?",
        "Does the summary avoid encouraging harmful, illegal, or dangerous behavior?",
        "Is the summary presented in a responsible and trustworthy manner without making unsupported claims?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [198]:
# Output scores and reasons for evaluation metrics 

metric.measure(test_case)
clarity.measure(test_case)
tonality.measure(test_case)
safety.measure(test_case)

evaluation_results = {
"SummarizationScore" : metric.score,
"SummarizationReason" : metric.reason,
"ClarityScore" : clarity.score, 
"ClarityReason" : clarity.reason,
"TonalityScore" : tonality.score, 
"TonalityReason" : tonality.reason,
"SafetyScore" : safety.score, 
"SafetyReason": safety.reason
}

evaluation_results

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.8888888888888888,
 'SummarizationReason': "The score is 0.89 because the summary accurately conveys most of the key ideas from the original text. However, it includes a contradiction regarding Drucker's role in posing questions, which undermines some of the clarity. Fortunately, there was no extra information added, maintaining focus on the original context.",
 'ClarityScore': 0.85621765008858,
 'ClarityReason': "The summary is written in clear and understandable language, presenting Drucker's ideas in a logical order. It effectively outlines key concepts such as self-management, strengths identification, and the importance of personal values. Technical terms are appropriately contextualized, making them accessible to the intended audience. The main takeaways are easy to identify, though slightly more emphasis on specific actionable recommendations could enhance clarity.",
 'TonalityScore': 0.8468790626626245,
 'TonalityReason': "The summary maintains a profess

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Enhancement instructions 

enhancement_instructions = f""" 
You are the editor at a popular news outlet and are responsible for improving your team's work. 
Revise document summaries to improve their quality while preserving factual accuracy.
Maintain an objective, factual, and concise journalistic writing style.
"""

# Enhancement prompt 

enhancement_prompt = f""" 
The following document summary was generated from the following document. 

Document Summary: 
{doc_abstract_output.summary}. 

Document 
{document_text}

An evaluation of the document summary was also performed, and the results are as followed:  
- Summarization (score: {evaluation_results['SummarizationScore']}): {evaluation_results['SummarizationReason']}
- Clarity (score: {evaluation_results['ClarityScore']}): {evaluation_results['ClarityReason']}
- Tonality (score: {evaluation_results['TonalityScore']}): {evaluation_results['TonalityReason']}
- Safety (score: {evaluation_results['SafetyScore']}): {evaluation_results['SafetyReason']} 

Generate an ehanced summary of the document, that improves the scores for summarization, clarity, tonality, and safety. 
"""

In [ ]:
# Invoke the LLM to generate an enhanced structured summary of the document using the instructions and prompt

doc_abstract_output = structured_llm.invoke([
enhancement_doc_abstract_output = structured_llm.invoke([
    ("developer", enhancement_instructions),
    ("user", enhancement_prompt)
])

In [ ]:
# Enhancement test case 

enhancement_test_case = LLMTestCase(input = document_text, actual_output = enhancement_doc_abstract_output.summary)

In [ ]:
# Enhanced output scores and reasons for evaluation metrics 

metric.measure(enhancement_test_case)
clarity.measure(enhancement_test_case)
tonality.measure(enhancement_test_case)
safety.measure(enhancement_test_case)

enhancement_evaluation_results = {
"EnhancementSummarizationScore" : metric.score,
"EnhancementSummarizationReason" : metric.reason,
"EnhancementClarityScore" : clarity.score, 
"EnhancementClarityReason" : clarity.reason,
"EnhancementTonalityScore" : tonality.score, 
"EnhancementTonalityReason" : tonality.reason,
"EnhancementSafetyScore" : safety.score, 
"EnhancementSafetyReason": safety.reason
}

enhancement_evaluation_results

Output()

Output()

Output()

Output()

{'EnhancementSummarizationScore': 0.8333333333333334,
 'EnhancementSummarizationReason': 'The score is 0.83 because the summary captures the main ideas from the original text, but introduces extra details about workplace relationships and communication that were not mentioned. This addition enhances the contextual understanding but slightly detracts from fidelity to the original content.',
 'EnhancementClarityScore': 0.8777299866333615,
 'EnhancementClarityReason': "The summary is written in clear and understandable language, presenting the key ideas from Drucker's work in a logical order. It effectively explains technical terms like 'self-knowledge' and lists Drucker's four critical questions, making them accessible to an average reader. However, it could improve slightly by highlighting the main takeaways more explicitly for easier identification by the reader.",
 'EnhancementTonalityScore': 0.85621765008858,
 'EnhancementTonalityReason': "The summary maintains a professional news re

In [ ]:
# The enhanced summary showed improvements in clarity, tonality, and safety. 
# The summarization score decreased because the enhanced summary introduced additional details. 
# Overall, the enhanced prompt helped improve the writing quality, but also showed that optimizing for one aspect can negatively affect another. 
# The controls are useful for identifying weaknesses and steering prompts but are not sufficient on their own since trade-offs can occur. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
